In [8]:
!pip uninstall -y pillow
!pip install pillow==10.3.0

!pip install -U transformers accelerate sentencepiece

Found existing installation: pillow 10.3.0
Uninstalling pillow-10.3.0:
  Successfully uninstalled pillow-10.3.0
Defaulting to user installation because normal site-packages is not writeable
  Using cached pillow-10.3.0-cp310-cp310-manylinux_2_28_x86_64.whl (4.5 MB)
Defaulting to user installation because normal site-packages is not writeable


In [9]:
from PIL import Image
print(Image.__version__)
print(hasattr(Image, "Resampling"))

10.3.0
True


In [10]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "klyang/MentaLLaMA-chat-7B"

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

print("Model loaded successfully")

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 291/291 [00:01<00:00, 287.51it/s]
Some parameters are on the meta device because they were offloaded to the cpu.


Model loaded successfully


[transformers] Error during conversion: ReadTimeout('The read operation timed out')


In [11]:
import pandas as pd

df = pd.read_csv("broken_100_rows.csv")

print(df.shape)
df.head()

(100, 4)


,instruction,input,Broken English(West African),output
0,You are a helpful mental health counselling as...,NaN,I dey feel anxious and on edge for some weeks ...,I understand that you're feeling anxious and o...
1,You are a helpful mental health counselling as...,I've been feeling guilty for not being able to...,I dey feel guilty say I no dey do enough for m...,I understand that you're feeling guilty for no...
2,You are a helpful mental health counselling as...,I've been grappling with the aftermath of a tr...,I dey deal with the effect of a traumatic even...,Your experience with a traumatic event is undo...
3,You are a helpful mental health counselling as...,She's been maintaining a stable vital sign rat...,"She dey stable, vital signs dey okay, and appe...",It's essential to acknowledge the progress you...
4,You are a helpful mental health counselling as...,I've been feeling lost and uncertain about my ...,I dey feel lost about my future and I even thi...,I empathize with your feelings of uncertainty ...


In [12]:
print(df.columns.tolist())

['instruction', 'input', 'Broken English(West  African)', 'output']


In [13]:
def build_prompt(user_text):

    return f"""You are a helpful, safe, and empathetic mental health assistant.

Respond ONLY with the assistant's reply.
Do NOT repeat the user input.
Do NOT repeat instructions.

User message:
{user_text}

Assistant response:
"""

In [14]:
def clean_response(text, prompt):

    if prompt in text:
        text = text.replace(prompt, "")

    prefixes = [
        "Assistant response:",
        "User message:",
        "User input:",
        "You are a helpful"
    ]

    for p in prefixes:
        if text.strip().startswith(p):
            text = text.split(p, 1)[-1]

    return text.strip()

In [15]:
def generate_response(prompt):

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.6,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    decoded = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return clean_response(decoded, prompt)

In [16]:
sample_prompt = build_prompt(
    df.iloc[0]["Broken English(West  African)"]
)

response = generate_response(sample_prompt)

print(response)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


I understand that you are feeling anxious and on edge, and that you are having difficulty focusing and sleeping. I also understand that you are avoiding social situations due to fear of judgment or criticism. It's important to address these feelings and seek help. Consider talking to a therapist, as they can provide you with the support and guidance you need. Additionally, there are many resources available online that can help you with anxiety and mental health. Don't let concerns about cost or stigma stop you from seeking help, as it is important to prioritize your well-being.


In [ ]:
broken_outputs = []

for i, row in df.iterrows():

    prompt = build_prompt(
        row["Broken English(West  African)"]
    )

    response = generate_response(prompt)

    broken_outputs.append(response)

    if (i + 1) % 10 == 0:
        print(f"Completed {i+1}/100")

[transformers] Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

Completed 10/100


[transformers] Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

Completed 20/100


[transformers] Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

...

[transformers] Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

Completed 80/100


[transformers] Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

Completed 90/100


[transformers] Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

In [ ]:
df["broken_outputs"] = broken_outputs

In [ ]:
df.to_csv("broken_outputs.csv", index=False)

print("Saved broken_outputs.csv")